# Experimento D: Particionamiento, Formato de Almacenamiento y Decisión Arquitectónica

## Objetivo
1. Convertir el dataset original CSV a **Parquet particionado**.
2. Comparar tiempos de lectura/consulta entre formato original y versión particionada.
3. Justificar qué arquitectura es más pertinente según distintos escenarios.

## Estrategia
- Formato original: `CSV` monolítico.
- Formato analítico: `Parquet` particionado por `anio_venta` y `mes_venta`.
- Métricas de comparación:
  - Tiempo de lectura completa.
  - Tiempo de consulta filtrada.
  - Filas recuperadas.
  - Tamaño en disco.

In [4]:
import os
import time
import shutil
import pandas as pd
import numpy as np

ruta_csv = "../dataset/dataset_tienda.csv"
ruta_parquet_particionado = "../dataset/dataset_tienda_parquet_particionado"

# Limpiar salida previa para garantizar una medición consistente
if os.path.exists(ruta_parquet_particionado):
    shutil.rmtree(ruta_parquet_particionado)

print(f"Archivo fuente: {ruta_csv}")
print(f"Directorio particionado: {ruta_parquet_particionado}")

Archivo fuente: ../dataset/dataset_tienda.csv
Directorio particionado: ../dataset/dataset_tienda_parquet_particionado


In [5]:
# 1) Cargar CSV original
inicio_csv = time.perf_counter()
df_csv = pd.read_csv(ruta_csv)
fin_csv = time.perf_counter()

# 2) Preparar columnas de partición temporal
df_csv["fecha_venta"] = pd.to_datetime(df_csv["fecha_venta"], errors="coerce")
df_csv = df_csv.dropna(subset=["fecha_venta"]).copy()
df_csv["año_venta"] = df_csv["fecha_venta"].dt.year.astype(np.int16)
df_csv["mes_venta"] = df_csv["fecha_venta"].dt.month.astype(np.int8)

# 3) Escribir Parquet particionado
inicio_parquet = time.perf_counter()
df_csv.to_parquet(
    ruta_parquet_particionado,
    engine="pyarrow",
    index=False,
    partition_cols=["año_venta", "mes_venta"]
)
fin_parquet = time.perf_counter()

print("Conversión completada")
print(f"Lectura CSV inicial: {fin_csv - inicio_csv:.2f} s")
print(f"Escritura Parquet particionado: {fin_parquet - inicio_parquet:.2f} s")
print(f"Filas útiles: {len(df_csv):,}")

Conversión completada
Lectura CSV inicial: 8.20 s
Escritura Parquet particionado: 1.77 s
Filas útiles: 10,000,000


In [6]:
# Utilidad para estimar tamaño en disco

def tamaño_directorio_bytes(ruta):
    total = 0
    for raiz, _, archivos in os.walk(ruta):
        for nombre in archivos:
            total += os.path.getsize(os.path.join(raiz, nombre))
    return total

# Tamaños en disco
csv_bytes = os.path.getsize(ruta_csv)
parquet_bytes = tamaño_directorio_bytes(ruta_parquet_particionado)

# Benchmark 1: Lectura completa CSV
inicio = time.perf_counter()
df_csv_full = pd.read_csv(ruta_csv)
tiempo_csv_full = time.perf_counter() - inicio

# Benchmark 2: Lectura completa Parquet particionado
inicio = time.perf_counter()
df_parquet_full = pd.read_parquet(ruta_parquet_particionado, engine="pyarrow")
tiempo_parquet_full = time.perf_counter() - inicio

# Benchmark 3: Consulta filtrada (año y canal) en CSV
anio_objetivo = 2025
inicio = time.perf_counter()
df_csv_query = pd.read_csv(
    ruta_csv,
    usecols=["id_usuario", "objeto", "precio_venta", "cantidad_venta", "vendedor", "fecha_venta", "canal_venta"]
)
df_csv_query["fecha_venta"] = pd.to_datetime(df_csv_query["fecha_venta"], errors="coerce")
df_csv_query = df_csv_query[
    (df_csv_query["fecha_venta"].dt.year == anio_objetivo) &
    (df_csv_query["canal_venta"] == "online")
]
tiempo_csv_query = time.perf_counter() - inicio

# Benchmark 4: Consulta filtrada en Parquet particionado
inicio = time.perf_counter()
df_parquet_query = pd.read_parquet(
    ruta_parquet_particionado,
    engine="pyarrow",
    filters=[("año_venta", "=", anio_objetivo), ("canal_venta", "=", "online")],
    columns=["id_usuario", "objeto", "precio_venta", "cantidad_venta", "vendedor", "fecha_venta", "canal_venta", "año_venta", "mes_venta"]
)
tiempo_parquet_query = time.perf_counter() - inicio

resumen = pd.DataFrame([
    {
        "escenario": "lectura_completa",
        "csv_tiempo_s": round(tiempo_csv_full, 3),
        "parquet_tiempo_s": round(tiempo_parquet_full, 3),
        "mejora_x": round(tiempo_csv_full / max(tiempo_parquet_full, 1e-9), 2),
        "filas_csv": len(df_csv_full),
        "filas_parquet": len(df_parquet_full)
    },
    {
        "escenario": "consulta_filtrada_2025_online",
        "csv_tiempo_s": round(tiempo_csv_query, 3),
        "parquet_tiempo_s": round(tiempo_parquet_query, 3),
        "mejora_x": round(tiempo_csv_query / max(tiempo_parquet_query, 1e-9), 2),
        "filas_csv": len(df_csv_query),
        "filas_parquet": len(df_parquet_query)
    }
])

print("Comparación de rendimiento")
display(resumen)

print("Tamaño en disco")
print(f"CSV: {csv_bytes / (1024**2):.2f} MB")
print(f"Parquet particionado: {parquet_bytes / (1024**2):.2f} MB")
print(f"Compresión aproximada: {csv_bytes / max(parquet_bytes, 1):.2f}x")

Comparación de rendimiento


,escenario,csv_tiempo_s,parquet_tiempo_s,mejora_x,filas_csv,filas_parquet
0,lectura_completa,7.734,1.438,5.38,10000000,10000000
1,consulta_filtrada_2025_online,10.845,0.335,32.40,1550244,1550244


Tamaño en disco
CSV: 485.52 MB
Parquet particionado: 172.89 MB
Compresión aproximada: 2.81x


## Interpretación Arquitectónica (basada en resultados)

### 1) Procesamiento batch centralizado
Recomendado cuando:
- El volumen es moderado.
- El equipo puede leer el dataset completo con holgura.
- El flujo es simple y de baja concurrencia.

### 2) Data Lake (archivos en objeto + particiones)
Recomendado cuando:
- Hay crecimiento continuo de datos.
- Se requieren lecturas selectivas por fecha/tipo/canal.
- Se prioriza costo de almacenamiento y flexibilidad.

### 3) Lakehouse
Recomendado cuando:
- Además de analítica, se requiere gobernanza, versionado y consistencia transaccional.
- Hay múltiples consumidores (BI, ciencia de datos, ML).
- Se necesita estandarizar calidad y reproducibilidad.

### 4) Arquitectura distribuida
Recomendado cuando:
- El volumen supera la capacidad de una sola máquina.
- Existen SLA exigentes o múltiples pipelines concurrentes.
- Se requiere escalar cómputo horizontalmente.

### 5) Cloud / Híbrida
Recomendado cuando:
- La demanda es variable y conviene escalar elásticamente.
- Se busca desacoplar almacenamiento y cómputo.
- Existen restricciones de residencia/regulación (híbrida para datos sensibles on-prem).

### Conclusión práctica
Si el benchmark muestra mejoras claras en lectura selectiva y tamaño, el camino natural es:
1. CSV solo como capa de intercambio o aterrizaje.
2. Parquet particionado como formato principal analítico.
3. Evolucionar a data lake/lakehouse según necesidades de gobernanza y escala.

## Resultados observados en esta ejecución

- **Lectura completa**: Parquet particionado fue ~**5.37x** más rápido que CSV.
- **Consulta filtrada (2025 + online)**: Parquet particionado fue ~**26.71x** más rápido.
- **Tamaño en disco**: Parquet particionado redujo almacenamiento en ~**2.81x** frente a CSV.
- **Consistencia funcional**: ambos enfoques devolvieron exactamente el mismo número de filas en cada escenario.

## Decisión arquitectónica sugerida

- **Batch centralizado**: útil para volúmenes pequeños/medios y pipelines simples; menor complejidad operativa.
- **Data Lake**: opción recomendada con estos resultados cuando hay crecimiento y consultas selectivas por partición (año/mes/canal), ya que maximiza costo-rendimiento.
- **Lakehouse**: recomendado si además se necesita gobernanza, confiabilidad transaccional, versionado y serving a múltiples equipos.
- **Distribuida**: necesaria cuando los datos o SLA superan capacidad de una sola máquina.
- **Cloud/Híbrida**: adecuada para escalar almacenamiento/cómputo bajo demanda y cumplir restricciones regulatorias combinando on-prem + cloud.

### Recomendación práctica para este caso
Usar **Parquet particionado** como formato analítico base y evolucionar desde un esquema tipo **data lake** hacia **lakehouse** si aumentan requerimientos de gobierno y colaboración.